# 🏛️ SF Landmarks - OCR + Ingest
### Powered by Elasticsearch Serverless + Mistral

San Francisco has designated **~390 city landmarks** since 1967: Mission Dolores, the Castro Theatre, Sutro Baths, a Chinatown bank, a Victorian on a quiet street. Each one has a **designation report** (a PDF, often a typewritten scan from decades ago) telling the building's history, plus a photo. The history is locked in those scans, which makes this a real-world **Mistral OCR** project.

**What this notebook does:**
1. Fetches the landmark list from [DataSF](https://data.sf.gov/d/97yj-54sx) - no API key required
2. Runs the first few pages of each designation report through **Mistral OCR** (`mistral-ocr-latest`) - you set how many
3. Indexes the landmarks + OCR text into `sf_landmarks`, so you can search San Francisco history by what the reports actually say

> **No Mistral key yet?** The notebook still runs: it skips the OCR step and indexes the landmark metadata, so you can add OCR later.

---
> **Data source:** Landmarks Listed in Article 10 of the San Francisco Planning Code, dataset `97yj-54sx`, from SF Planning. Designation reports and photos are linked from each record.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import requests
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless + set your Mistral key
> **Elastic Cloud Console → Your Project → Connection Details.** The Mistral key is handed out at the event.

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'
MISTRAL_API_KEY  = '<MISTRAL_API_KEY>'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Fetch the landmarks
Each landmark has a polygon outline; we also compute a centre point so you can run distance queries.

In [ ]:
resp = requests.get('https://data.sf.gov/resource/97yj-54sx.json', params={'$limit': 5000}, timeout=60)
resp.raise_for_status()
rows = resp.json()

def centroid(geom):
    """Rough centre of a (Multi)Polygon: the average of its outer-ring points."""
    if not geom:
        return None
    polys = geom['coordinates'] if geom['type'] == 'MultiPolygon' else [geom['coordinates']]
    pts = [p for poly in polys for p in poly[0]]
    if not pts:
        return None
    return {'lat': sum(p[1] for p in pts) / len(pts), 'lon': sum(p[0] for p in pts) / len(pts)}

def to_int(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None

landmarks = []
for r in rows:
    doc = {
        'landmark_no':     to_int(r.get('landmarkno')),
        'name':            r.get('name'),
        'address':         r.get('address'),
        'apn':             r.get('apn'),
        'year_designated': to_int(r.get('yeardesignated')) or None,   # 0 means "pending"
        'document_url':    (r.get('designationdocument') or {}).get('url'),
        'photo_url':       (r.get('photo') or {}).get('url'),
        'location':        centroid(r.get('the_geom')),
    }
    landmarks.append({k: v for k, v in doc.items() if v not in (None, '')})

with_docs = [l for l in landmarks if l.get('document_url')]
print(f'✅ Fetched {len(landmarks)} landmarks ({len(with_docs)} with a designation report)')
for l in sorted(with_docs, key=lambda l: l.get('landmark_no') or 0)[:5]:
    print(f"  #{l.get('landmark_no')}: {l['name']} ({l.get('year_designated')})")

## 3. OCR the designation reports with Mistral 🔎
We hand Mistral OCR each report's URL and ask for the first `MAX_PAGES` pages. The response is markdown, one block per page.

> The older reports are typewritten scans from the 1960s-80s; the newer ones are 50-page born-digital PDFs. Start with a small `OCR_LIMIT` (it takes a few seconds per report), then raise it. For the photos, try a vision-capable Mistral model instead.

In [ ]:
OCR_LIMIT = 20   # how many reports to OCR; raise toward ~350 for all of them
MAX_PAGES = 3    # pages per report (the history is usually up front)

def mistral_ocr(url):
    r = requests.post('https://api.mistral.ai/v1/ocr',
        headers={'Authorization': f'Bearer {MISTRAL_API_KEY}', 'Content-Type': 'application/json'},
        json={'model': 'mistral-ocr-latest',
              'document': {'type': 'document_url', 'document_url': url},
              'pages': list(range(MAX_PAGES))},
        timeout=180)
    r.raise_for_status()
    return '\n\n'.join(p.get('markdown', '') for p in r.json().get('pages', [])).strip()

if MISTRAL_API_KEY.startswith('<'):
    print('⏭️  No Mistral key set - skipping OCR. Set MISTRAL_API_KEY above and re-run this cell to add it.')
else:
    # Oldest landmarks first: their reports are the scans that really need OCR
    for l in sorted(with_docs, key=lambda l: l.get('landmark_no') or 9999)[:OCR_LIMIT]:
        try:
            l['report_text'] = mistral_ocr(l['document_url'])
        except Exception as e:
            print(f"  ! OCR failed for #{l.get('landmark_no')} {l['name']}: {e}")
            continue
        print(f"  #{l.get('landmark_no'):<4} {l['name'][:40]:40s} → {len(l['report_text'])} chars")
    print('✅ OCR complete')

## 4. Create index and ingest

In [ ]:
INDEX = 'sf_landmarks'

mapping = {'mappings': {'properties': {
    'landmark_no':     {'type': 'integer'},
    'name':            {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'address':         {'type': 'text'},
    'apn':             {'type': 'keyword'},   # assessor's parcel number
    'year_designated': {'type': 'integer'},
    'document_url':    {'type': 'keyword'},
    'photo_url':       {'type': 'keyword'},
    'report_text':     {'type': 'text'},      # Mistral OCR output
    'location':        {'type': 'geo_point'},
}}}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')
es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

docs = landmarks
success, errors = helpers.bulk(es, [{'_index': INDEX, '_source': d} for d in docs], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {success} documents into {INDEX} ({len(errors)} errors)')
print(f"   Landmarks with OCR text: {sum(1 for d in docs if 'report_text' in d)}")

## 5. Verify

In [ ]:
resp = es.search(index=INDEX, body={'size': 0, 'aggs': {
    'by_decade': {'histogram': {'field': 'year_designated', 'interval': 10}},
}})
print(f"Total landmarks: {resp['hits']['total']['value']}\n")
print('Designations by decade:')
for b in resp['aggregations']['by_decade']['buckets']:
    print(f"  {int(b['key'])}s  {'█' * (b['doc_count'] // 3)} {b['doc_count']}")

# Full-text search over the OCR'd history (only returns hits if you ran step 3)
resp = es.search(index=INDEX, body={'size': 5,
    'query': {'match': {'report_text': 'earthquake fire 1906'}},
    '_source': ['landmark_no', 'name', 'address'],
})
print(f"\nReports mentioning the 1906 earthquake and fire: {resp['hits']['total']['value']}")
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"  #{s['landmark_no']} {s['name']} - {s.get('address')}")

## 6. Going further with Mistral

- **Ask the archive:** add a `semantic_text` field on `report_text` backed by Mistral embeddings and ask "which buildings were designed by women architects?" or "landmarks tied to the labor movement". See [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md).
- **Structured extraction:** use Mistral **Document AI** / structured outputs to pull architect, year built, original use, and style out of each report, then aggregate on them.
- **History near you:** combine `location` with a geo query to tell a visitor the story of the three landmarks closest to where they're standing, read aloud with **Voxtral TTS**.
- **Photo check:** send `photo_url` to a vision-capable Mistral model and compare what it sees with the report's description.

See [mistral_guide.md](mistral_guide.md) for calling Mistral directly.

---
## ✅ Index loaded
`sf_landmarks` holds San Francisco's landmarks and the OCR'd text of their designation reports.

**To OCR more reports**, raise `OCR_LIMIT` / `MAX_PAGES` and re-run cells 3-4. The index is dropped and recreated each time, so there are no duplicates.